# COS30049 A2 - Stage B3: the dev tier on DS7 v2's validation prompts for round 1, A and B (T4)

The B3 trial (`cos30049a2-text150m-b3-trial`) scored its three writers on the weak-case dev slice, but its run on DS7
v2's validation prompts stopped: those rows carry no template. `lm/eval_explain.py` now judges such rows against
their reference reply (same reasons in the same order; every strict measure identical on all 1,637 DS7 v2 test
rows). This notebook runs only those evaluations, one after another on one GPU, with the trial's saved models.

**Settings:** Accelerator **GPU T4 x2** (one is used), **Internet on**. **Inputs:** this account's dataset
`cos30049a2-text150m-base` and notebooks `cos30049a2-text150m-stage-data` and `cos30049a2-text150m-b3-trial`.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "25f18cb5aebade4cc5d8335558148c8d967146c1"   # the code this run uses; recorded in kaggle_run.json
PINS = ["tokenizers==0.21.1", "numpy==2.2.6"]  # environment.yml; torch is Kaggle's own (recorded)

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU in this session: Settings panel -> Accelerator -> GPU T4 x2 (needs a verified phone).")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

def find_input(rel):
    """The file under /kaggle/input whose path ends with rel (inputs mount under their own folder names)."""
    hits = [p for p in Path("/kaggle/input").rglob(Path(rel).name) if p.as_posix().endswith(rel)]
    if not hits:
        raise SystemExit(f"{rel} is not in any input: add the inputs named at the top")
    return hits[0]


def check_base():
    """The round-1 files (dataset cos30049a2-text150m-base), each checked against the dataset's SHA256SUMS.txt."""
    sums = find_input("SHA256SUMS.txt")
    for line in sums.read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        assert sha256(sums.parent / name.lstrip("*")) == digest, f"{name} differs from SHA256SUMS.txt"
    return sums.parent

In [ ]:
base = check_base()
stage_manifest_path = find_input("stage_manifest.json")
for rel, digest in json.loads(stage_manifest_path.read_text(encoding="utf-8"))["files"].items():
    assert sha256(stage_manifest_path.parent / rel) == digest, f"{rel} differs from stage_manifest.json"
STAGE = stage_manifest_path.parent / "stage"
WRITERS = [("text150m_r1", base / "text150m_explain_v1.pt"),
           ("text150m_b3a", find_input("runs/text-150m-explain-b3a/model_best.pt")),
           ("text150m_b3b", find_input("runs/text-150m-explain-b3b/model_best.pt"))]

In [ ]:
# The writer's dev tier (TRAINING_PLAN section 2; settings weakCases): the weak-case dev slice and devVal prompts of
# DS7 v2's validation split. Never the test sets: those are scored once per frozen candidate.
WEAK_SETTINGS = json.loads((SRC / "config" / "settings.json").read_text(encoding="utf-8"))["weakCases"]
DEV = [(STAGE / "dev" / "weak_dev.jsonl", "devweak", 100000), (STAGE / "dev" / "ds7v2_val.jsonl", "devval", WEAK_SETTINGS["devVal"])]


def dev_jobs(model, checkpoint):
    return [(model, checkpoint, test, tag, n, WEAK_SETTINGS["devValSeed"]) for test, tag, n in DEV]


def run_queue(gpu, queue):
    for model, checkpoint, test, tag, n, seed in queue:
        sh(sys.executable, "-m", "lm.eval_explain", "--model", model, "--checkpoint", str(checkpoint), "--test", str(test),
           "--tag", tag, "--n", str(n), "--seed", str(seed), log=f"eval-{model}-{tag}.log",
           env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))


def run_queues(queues):
    threads = [threading.Thread(target=run_queue, args=(gpu, q)) for gpu, q in enumerate(queues)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

# Only DS7 v2's validation prompts (the weak-case slice was scored by the trial); one queue, so the writers never
# write the shared summary table at the same time.
run_queue(0, [job for name, ckpt in WRITERS for job in dev_jobs(name, ckpt) if job[3] == "devval"])
tables = WORK / "reports" / "tables"
tables.mkdir(parents=True, exist_ok=True)
for f in (SRC / "reports" / "tables").glob("assistant_eval*_devval.csv"):
    shutil.copy(f, tables)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "python": sys.version.split()[0],
    "packages": {p: version(p) for p in ("torch", "numpy", "tokenizers")},
    "hours": round((time.time() - T0) / 3600, 2)}, indent=2), encoding="utf-8")